# 03 - Evaluación del Modelo y Explicabilidad
## FraudIA Claims

Este notebook evalúa el rendimiento del modelo y genera explicaciones para casos individuales.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.explainability.explain_score import explain_single_case, generate_executive_summary
from src.ai_agent.claims_agent import ClaimsAgent
from src.app.powerbi_export import export_to_powerbi

print('Módulos cargados')

In [ ]:
df = pd.read_csv('data/processed/siniestros_scored.csv')
print(f'Datos cargados: {df.shape}')
df.head()

## 1. Resumen Ejecutivo

In [ ]:
summary = generate_executive_summary(df)
print('RESUMEN EJECUTIVO')
print('=' * 50)
print(f"Total siniestros: {summary['total_siniestros']}")
print(f"\nDistribución Semáforo:")
for sem, count in summary['distribucion_semaforo'].items():
    pct = summary['porcentajes'][sem]
    print(f"  {sem}: {count} ({pct}%)")
print(f"\nScore promedio: {summary['score_promedio']}")
print(f"Score máximo: {summary['score_maximo']}")

## 2. Explicación de Casos Individuales

In [ ]:
score_col = 'score_hibrido' if 'score_hibrido' in df.columns else 'score_reglas'
top_5 = df.nlargest(5, score_col)

for _, row in top_5.iterrows():
    explanation = explain_single_case(row)
    print(f"\n{'='*60}")
    print(explanation['resumen'])
    print(f"{'='*60}")

## 3. Agente IA - Consultas

In [ ]:
agent = ClaimsAgent(df)

queries = [
    '¿Cuál es el resumen general?',
    '¿Cuáles son los 5 casos de mayor riesgo?',
    '¿Qué patrones se detectaron?',
    'Análisis por ramo',
    '¿Cuál es el monto total reclamado?',
]

for q in queries:
    result = agent.query(q)
    print(f"\nPregunta: {q}")
    print(f"Respuesta:\n{result['respuesta']}")
    print('-' * 50)

## 4. Exportación Power BI

In [ ]:
from src.ingestion.load_data import load_all_from_directory
datasets = load_all_from_directory('data/synthetic')
path = export_to_powerbi(datasets, df, 'data/processed/powerbi_export.xlsx')
print(f'Exportación Power BI: {path}')

In [ ]:
print('\nEvaluación completada. El sistema está listo para demo.')